In [11]:
import pandas as pd

print("Part A - Data Understanding")
data = pd.read_csv('iot_sensor_data_raw.csv')

print("\nStructure and dimension:", data.shape)

data["Timestamp"] = pd.to_datetime(data["Timestamp"], errors="coerce")

is_sorted = data["Timestamp"].is_monotonic_increasing
print("\nTimestamps sorted?", is_sorted)

missing_counts = data.isnull().sum()
print("\nMissing values per column:\n", missing_counts)

missing_percentage = (data.isnull().mean() * 100).round(2)
print("\nPercentage of missing values per column:\n", missing_percentage)

Part A - Data Understanding

Structure and dimension: (20000, 9)

Timestamps sorted? False

Missing values per column:
 Timestamp           0
Device_ID           0
Temperature       299
Humidity          300
Pressure          300
Vibration         298
Battery_Level       0
Location            0
Machine_Status      0
dtype: int64

Percentage of missing values per column:
 Timestamp         0.00
Device_ID         0.00
Temperature       1.50
Humidity          1.50
Pressure          1.50
Vibration         1.49
Battery_Level     0.00
Location          0.00
Machine_Status    0.00
dtype: float64


In [29]:
print("Part B - Data Cleaning")

data["Temperature"] = data["Temperature"].fillna(data["Temperature"].mean())
data["Vibration"]   = data["Vibration"].fillna(data["Vibration"].mean())
data["Battery_Level"] = data["Battery_Level"].fillna(data["Battery_Level"].mean())
temp_mean = data["Temperature"].mean()
temp_std  = data["Temperature"].std()

data["Temp_Abnormal"] = (data["Temperature"] < temp_mean - 3*temp_std) | \
                      (data["Temperature"] > temp_mean + 3*temp_std)

print(data[data["Temp_Abnormal"]])

vib_mean = data["Vibration"].mean()
vib_std = data["Vibration"].std()
data["Vibration_Abnormal"] = np.abs(data["Vibration"] - vib_mean) > (3 * vib_std)
print(data[data["Vibration_Abnormal"]])

critical_machines = data[data["Battery_Level"] < 10]
print("Machines with critically low battery levels:")
print(critical_machines[["Device_ID", "Battery_Level"]])

#Abnormal reading
#Battery less than 10
#Temperation between 0 and 100. Beyond 100 or below 0 is considered as abnormal


Part B - Data Cleaning
                Timestamp Device_ID  Temperature  Humidity  Pressure  \
77    2026-01-11 05:00:00   DEV_004   150.182051     64.84   1020.10   
337   2026-07-11 07:30:00   DEV_001   133.908634     50.20   1001.87   
596   2026-06-20 00:15:00   DEV_003   152.454296     75.08    983.70   
724   2026-07-17 01:30:00   DEV_007   150.099637     65.03   1017.66   
1460  2026-04-14 09:30:00   DEV_002   130.403716     46.97   1010.92   
...                   ...       ...          ...       ...       ...   
19567 2026-07-01 18:30:00   DEV_004   122.438182     45.92   1012.98   
19572 2026-02-26 02:45:00   DEV_001   143.801135     47.22   1009.58   
19772 2026-01-13 00:30:00   DEV_008   129.367099     50.78   1011.39   
19783 2026-01-22 08:15:00   DEV_005   152.974450     57.94   1003.31   
19950 2026-02-02 20:00:00   DEV_008   125.279609     78.80   1009.85   

       Vibration  Battery_Level   Location Machine_Status  Temp_Abnormal  \
77          3.59          41.32  Fac

In [39]:
print("Part c - Time-Series Analysis")

data["Hour"] = data["Timestamp"].dt.hour
avg_temp_by_hour = data.groupby("Hour")["Temperature"].mean()

avg_temp_device = data.groupby("Device_ID")["Temperature"].mean()
print(avg_temp_device)

avg_vib_device = data.groupby("Device_ID")["Vibration"].mean()
print(avg_vib_device)

max_temp_device = data.groupby("Device_ID")["Temperature"].max()
print(max_temp_device)

min_battery_device = data.groupby("Device_ID")["Battery_Level"].min()
print(min_battery_device)

highest_vib_device = avg_vib_device.idxmax()
print(highest_vib_device)


avg_temp_factory = data.groupby("Location")["Temperature"].mean()
highest_temp_factory = avg_temp_factory.idxmax()
print(highest_temp_factory)


Part c - Time-Series Analysis
Device_ID
DEV_001    70.263204
DEV_002    70.384507
DEV_003    69.931047
DEV_004    70.578075
DEV_005    70.109588
DEV_006    70.010784
DEV_007    70.143433
DEV_008    70.331719
Name: Temperature, dtype: float64
Device_ID
DEV_001    2.552664
DEV_002    2.561525
DEV_003    2.536610
DEV_004    2.522176
DEV_005    2.494705
DEV_006    2.552879
DEV_007    2.514962
DEV_008    2.526515
Name: Vibration, dtype: float64
Device_ID
DEV_001    159.746758
DEV_002    151.818550
DEV_003    159.746758
DEV_004    151.965273
DEV_005    152.974450
DEV_006    136.003203
DEV_007    153.877632
DEV_008    153.622123
Name: Temperature, dtype: float64
Device_ID
DEV_001    2.257059
DEV_002    2.154615
DEV_003    3.670604
DEV_004    8.514009
DEV_005    2.956738
DEV_006    2.201033
DEV_007    2.146086
DEV_008    2.011730
Name: Battery_Level, dtype: float64
DEV_002
Factory_C


In [42]:
print("Part D - Create New variables")

def battery_status(level):
    if level >= 50:
        return "Healthy"
    elif level >= 20:
        return "Moderate"
    else:
        return "Critical"

data["Battery_Status"] = data["Battery_Level"].apply(battery_status)
print(data["Battery_Status"])


def temp_status(temp):
    if temp < 0 or temp > 100:
        return "Critical"
    elif temp < 10 or temp > 80:
        return "Warning"
    else:
        return "Normal"
data["Temperature_Status"] = data["Temperature"].apply(temp_status)
print(data["Temperature_Status"])


def vib_status(vib):
    if vib > 10:
        return "Critical"
    elif vib > 5:
        return "Warning"
    else:
        return "Normal"
data["Vibration_Status"] = data["Vibration"].apply(vib_status)
print(data["Vibration_Status"])


def machine_health(row):
    if "Critical" in [row["Battery_Status"], row["Temperature_Status"], row["Vibration_Status"]]:
        return "Critical"
    elif "Warning" in [row["Battery_Status"], row["Temperature_Status"], row["Vibration_Status"]]:
        return "Warning"
    else:
        return "Normal"
data["Machine_Health"] = data.apply(machine_health, axis=1)
print(data["Machine_Health"])


Part D - Create New variables
0        Moderate
1         Healthy
2        Critical
3        Critical
4        Moderate
           ...   
19995    Moderate
19996    Critical
19997     Healthy
19998    Critical
19999     Healthy
Name: Battery_Status, Length: 20000, dtype: object
0         Normal
1         Normal
2        Warning
3         Normal
4         Normal
          ...   
19995     Normal
19996     Normal
19997     Normal
19998     Normal
19999     Normal
Name: Temperature_Status, Length: 20000, dtype: object
0        Normal
1        Normal
2        Normal
3        Normal
4        Normal
          ...  
19995    Normal
19996    Normal
19997    Normal
19998    Normal
19999    Normal
Name: Vibration_Status, Length: 20000, dtype: object
0          Normal
1          Normal
2        Critical
3        Critical
4          Normal
           ...   
19995      Normal
19996    Critical
19997      Normal
19998    Critical
19999      Normal
Name: Machine_Health, Length: 20000, dtype: object


In [46]:
print("Part E — Advanced Analysis")

critical_counts = data[data["Machine_Health"]=="Critical"].groupby("Device_ID").size()
devices_critical_5plus = critical_counts[critical_counts > 5]
print(devices_critical_5plus)

abnormal_counts = data[data["Machine_Health"]!="Normal"].groupby("Location").size()
highest_abnormal_factory = abnormal_counts.idxmax()
print("\n", highest_abnormal_factory)


device_total = data.groupby("Device_ID").size()
device_abnormal = data[data["Machine_Health"]!="Normal"].groupby("Device_ID").size()
device_abnormal_percentage = (device_abnormal / device_total * 100).round(2)
print("\n", device_abnormal_percentage)

highest_priority_device = device_abnormal_percentage.idxmax()
print("\n", highest_priority_device)


Part E — Advanced Analysis
Device_ID
DEV_001    304
DEV_002    312
DEV_003    277
DEV_004    323
DEV_005    319
DEV_006    305
DEV_007    270
DEV_008    296
dtype: int64

 Factory_B

 Device_ID
DEV_001    21.16
DEV_002    21.62
DEV_003    20.63
DEV_004    21.90
DEV_005    21.57
DEV_006    20.80
DEV_007    19.85
DEV_008    21.44
dtype: float64

 DEV_004
